# LeetCode 1540 — Can Convert String in K Moves bằng Python

Link đề: https://leetcode.com/problems/can-convert-string-in-k-moves/ · Độ khó: **Medium** · Chủ đề: Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def canConvertString(self, s: str, t: str, k: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Tính **độ dịch vòng** giữa hai chữ cái bằng `(ord(b) − ord(a)) % 26` và hiểu vì sao phải lấy modulo.
2. Nhận ra "dịch `i` lần" tương đương "dịch `i mod 26` lần" ⇒ các nước đi được **phân lớp theo số dư mod 26**.
3. Từ một mô phỏng "cấp phát nước đi nhỏ nhất còn trống" (baseline), rút ra công thức: lần thứ `c` cần độ dịch `d` phải dùng nước đi `d + 26·(c − 1)`.
4. Viết lời giải `O(n)` thời gian, `O(26)` bộ nhớ, có thể **dừng sớm**.
5. Cẩn thận các bẫy: độ dài khác nhau, độ dịch 0 không tốn nước đi, lệch một ở `c − 1`.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s`, `t` và số nguyên `k`. Muốn biến `s` thành `t` trong **không quá `k` nước đi**. Ở nước đi thứ `i` (`1 ≤ i ≤ k`), được chọn **một** trong hai:

- Chọn một chỉ số `j` của `s` **chưa từng được chọn** ở nước trước, và **dịch** ký tự `s[j]` đúng `i` lần.
- Không làm gì.

Dịch một lần nghĩa là thay bằng chữ cái kế tiếp (vòng: `'z'` → `'a'`). Mỗi chỉ số được chọn **tối đa một lần**. Trả về `True` nếu làm được.

**Ví dụ 1:** `s = "input"`, `t = "ouput"`, `k = 9` → `True` (nước 6 dịch `'i'` thành `'o'`, nước 7 dịch `'n'` thành `'u'`).

**Ví dụ 2:** `s = "abc"`, `t = "bcd"`, `k = 10` → `False` (cả ba ký tự cần dịch 1; chỉ nước 1 dịch đúng 1 trong phạm vi `k = 10` — các nước 27, 53 quá xa).

**Ví dụ 3:** `s = "aab"`, `t = "bbb"`, `k = 27` → `True` (nước 1 cho `'a'` đầu, nước 27 cho `'a'` thứ hai: 27 lần dịch ≡ 1).

**Ràng buộc:** `1 ≤ s.length, t.length ≤ 10⁵`, `0 ≤ k ≤ 10⁹`, chỉ chữ thường.

In [1]:
import random
import string
import time
from collections import Counter
from typing import List

examples = [("input", "ouput", 9, True), ("abc", "bcd", 10, False), ("aab", "bbb", 27, True)]
for s, t, k, want in examples:
    print(f"s={s!r:8} t={t!r:8} k={k:>2} -> mong đợi {want}")

s='input'  t='ouput'  k= 9 -> mong đợi True
s='abc'    t='bcd'    k=10 -> mong đợi False
s='aab'    t='bbb'    k=27 -> mong đợi True


Chú ý: đề cho phép `s` và `t` **khác độ dài** (ràng buộc ghi riêng `s.length`, `t.length`). Khi đó không thể biến đổi (thao tác không thêm/bớt ký tự) ⇒ `False`.

## 3. Kiến thức nền

### 3.1 `ord`, `chr` và mã chữ cái

`ord('a') = 97`. Chỉ số chữ cái `x` trong bảng (0..25) là `ord(x) − 97`; ngược lại `chr(97 + i)`.

In [2]:
print(ord("a"), ord("z"), chr(97 + 2), ord("o") - ord("i"))

97 122 c 6


### 3.2 Phép modulo với số âm trong Python

Trong Python, `a % 26` luôn nằm trong `[0, 25]` kể cả khi `a < 0` (khác C/Java). Nhờ vậy độ dịch vòng từ `a` tới `b` là `(ord(b) − ord(a)) % 26`.

In [3]:
for a, b in [("a", "b"), ("z", "a"), ("c", "a"), ("q", "q")]:
    print(f"{a} -> {b}: hiệu = {ord(b) - ord(a):>3}, dịch vòng = {(ord(b) - ord(a)) % 26}")
print("(-25) % 26 =", (-25) % 26)

a -> b: hiệu =   1, dịch vòng = 1
z -> a: hiệu = -25, dịch vòng = 1
c -> a: hiệu =  -2, dịch vòng = 24
q -> q: hiệu =   0, dịch vòng = 0
(-25) % 26 = 1


### 3.3 Dịch `i` lần tương đương dịch `i mod 26` lần

Bảng chữ cái có 26 chữ, dịch 26 lần quay về chỗ cũ. Hàm `shift(c, i)` minh hoạ.

In [4]:
def shift(c, i):
    return chr(97 + (ord(c) - 97 + i) % 26)

print(shift("a", 1), shift("a", 27), shift("a", 53), shift("z", 1), shift("i", 6))

b b b a o


### 3.4 `collections.Counter` và mảng đếm 26 ô

Đếm số ký tự cần mỗi độ dịch `d ∈ [1, 25]`: dùng `Counter` hoặc mảng `[0] * 26`.

In [5]:
s, t = "aab", "bbb"
diffs = [(ord(b) - ord(a)) % 26 for a, b in zip(s, t)]
print("độ dịch:", diffs, "| Counter:", Counter(d for d in diffs if d))
cnt = [0] * 26
for d in diffs:
    cnt[d] += 1
print("mảng đếm:", cnt[:5], "...")

độ dịch: [1, 1, 0] | Counter: Counter({1: 2})
mảng đếm: [1, 2, 0, 0, 0] ...


### 3.5 `zip` dừng ở chuỗi ngắn hơn

`zip("ab", "abc")` chỉ cho 2 cặp — dễ quên kiểm tra độ dài.

In [6]:
print(list(zip("ab", "abc")))

[('a', 'a'), ('b', 'b')]


## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 1 `"input" → "ouput"`: độ dịch từng vị trí:

| j | s[j] | t[j] | dịch |
|---|------|------|------|
| 0 | i | o | 6 |
| 1 | n | u | 7 |
| 2 | p | p | 0 (không cần) |
| 3 | u | u | 0 |
| 4 | t | t | 0 |

Cần một nước đi có số lần dịch ≡ 6 (mod 26) và một nước ≡ 7. Nhỏ nhất là nước 6 và nước 7 ≤ 9 ⇒ `True`.

Với ví dụ 2: ba vị trí cùng cần dịch 1. Các nước có số dư 1 là **1, 27, 53, …** Ba vị trí cần ba nước **khác nhau** ⇒ nước lớn nhất phải dùng là 53 > 10 ⇒ `False`.

Tổng quát: nếu có `c` vị trí cần độ dịch `d`, ta phải dùng các nước `d, d + 26, …, d + 26(c−1)`; điều kiện là `d + 26(c − 1) ≤ k`.

In [7]:
for s, t, k, want in examples:
    diffs = [(ord(b) - ord(a)) % 26 for a, b in zip(s, t)]
    cnt = Counter(d for d in diffs if d)
    need = {d: d + 26 * (c - 1) for d, c in cnt.items()}
    print(f"{s}->{t}: độ dịch {diffs} | nước lớn nhất cần theo từng d: {need} | k={k}")

input->ouput: độ dịch [6, 7, 0, 0, 0] | nước lớn nhất cần theo từng d: {6: 6, 7: 7} | k=9
abc->bcd: độ dịch [1, 1, 1] | nước lớn nhất cần theo từng d: {1: 53} | k=10
aab->bbb: độ dịch [1, 1, 0] | nước lớn nhất cần theo từng d: {1: 27} | k=27


## 5. Cách 1 — Brute Force / baseline: cấp phát nước đi bằng tập "đã dùng"

Xét lần lượt các vị trí cần dịch `d > 0`. Với mỗi vị trí, tìm nước đi **nhỏ nhất** chưa dùng có số lần dịch `≡ d (mod 26)`: thử `d, d + 26, d + 52, …`. Nếu nước tìm được `> k` thì thất bại.

In [8]:
def brute_assign(s, t, k):
    if len(s) != len(t):
        return False
    used = set()
    for a, b in zip(s, t):
        d = (ord(b) - ord(a)) % 26
        if d == 0:
            continue
        m = d
        while m in used:          # tìm nước nhỏ nhất ≡ d chưa dùng
            m += 26
        if m > k:
            return False
        used.add(m)
    return True

for s, t, k, want in examples:
    print(s, t, k, "->", brute_assign(s, t, k), "mong đợi", want)

input ouput 9 -> True mong đợi True
abc bcd 10 -> False mong đợi False
aab bbb 27 -> True mong đợi True


**Độ phức tạp:** nếu `c` vị trí cùng một độ dịch, vòng `while` lần lượt chạy `0, 1, …, c−1` bước ⇒ `O(c²)`; worst case mọi vị trí cùng độ dịch: `O(n²)` (≈ `5·10⁹` với `n = 10⁵`). Space `O(n)`.

**Nút thắt:** vòng `while` "nhảy" qua các nước đã dùng — nhưng các nước đã dùng của lớp `d` **luôn** là `d, d+26, …` liên tiếp, nên chỉ cần **đếm** số lần đã dùng.

In [9]:
s_bad = "a" * 6000
t_bad = "b" * 6000
t0 = time.perf_counter()
r = brute_assign(s_bad, t_bad, 10**9)
print(f"n=6000, mọi vị trí cùng d=1: {r} ({time.perf_counter() - t0:.2f}s) — tăng n lên 10^5 sẽ chậm hơn ~280 lần")

n=6000, mọi vị trí cùng d=1: True (0.65s) — tăng n lên 10^5 sẽ chậm hơn ~280 lần


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** các nước đi phân thành 26 lớp theo `i mod 26`; lớp `d` (1 ≤ d ≤ 25) gồm `d, d+26, d+52, …`. Mỗi vị trí cần độ dịch `d` phải lấy một nước **riêng** trong lớp `d`. Lớp `0` (nước 26, 52, …) chỉ có ích cho vị trí không cần dịch — mà vị trí đó cứ để yên là được.

**Quy tắc:**
- **Nếu** `len(s) != len(t)` **thì** `False`.
- Gọi `cnt[d]` = số vị trí cần độ dịch `d`. **Nếu** với mọi `d ∈ [1, 25]`: `cnt[d] == 0` hoặc `d + 26·(cnt[d] − 1) ≤ k` **thì** `True`, ngược lại `False`.

**Dạng dừng sớm:** duyệt từng vị trí, tăng `cnt[d]` rồi kiểm tra ngay `d + 26·(cnt[d] − 1) > k` ⇒ trả `False`.

**Invariant:** sau khi xét `j` vị trí, `cnt[d]` = số nước của lớp `d` đã cấp phát, và các nước đó chính là `d, d+26, …, d+26(cnt[d]−1)` — đúng như mô phỏng baseline.

## 7. Các cách tiếp theo

### Cách 2 — Đếm theo độ dịch rồi kiểm tra từng lớp

Từng bước: kiểm tra độ dài → đếm `cnt[d]` → với mỗi `d` có `cnt[d] > 0`, so nước lớn nhất cần dùng với `k`.

In [10]:
def count_then_check(s, t, k):
    if len(s) != len(t):
        return False
    cnt = [0] * 26
    for a, b in zip(s, t):
        cnt[(ord(b) - ord(a)) % 26] += 1
    for d in range(1, 26):
        if cnt[d] and d + 26 * (cnt[d] - 1) > k:
            return False
    return True

for s, t, k, want in examples:
    print(s, t, k, "->", count_then_check(s, t, k), "mong đợi", want)

input ouput 9 -> True mong đợi True
abc bcd 10 -> False mong đợi False
aab bbb 27 -> True mong đợi True


In [11]:
class SolutionCount:
    def canConvertString(self, s: str, t: str, k: int) -> bool:
        if len(s) != len(t):
            return False
        cnt = [0] * 26
        for a, b in zip(s, t):
            cnt[(ord(b) - ord(a)) % 26] += 1
        return all(cnt[d] == 0 or d + 26 * (cnt[d] - 1) <= k for d in range(1, 26))

print([SolutionCount().canConvertString(s, t, k) for s, t, k, _ in examples])

[True, False, True]


**Độ phức tạp:** Time `O(n + 26)`, Space `O(26)`.

### Cách 3 — Kiểm tra ngay khi đếm (dừng sớm)

Thay vì đếm xong mới kiểm tra, kiểm tra ngay lúc tăng bộ đếm: lần thứ `c` gặp độ dịch `d` dùng nước `d + 26(c−1)`. Tương đương cách 2 nhưng có thể dừng sớm.

In [12]:
class SolutionEarly:
    def canConvertString(self, s: str, t: str, k: int) -> bool:
        if len(s) != len(t):
            return False
        used = [0] * 26                  # used[d] = số nước lớp d đã dùng
        for a, b in zip(s, t):
            d = (ord(b) - ord(a)) % 26
            if d == 0:
                continue
            move = d + 26 * used[d]      # nước nhỏ nhất còn trống của lớp d
            if move > k:
                return False
            used[d] += 1
        return True

print([SolutionEarly().canConvertString(s, t, k) for s, t, k, _ in examples])

[True, False, True]


**Độ phức tạp:** Time `O(n)` worst, có thể sớm hơn; Space `O(26)`.

### Cách 4 — Đếm số nước khả dụng của mỗi lớp (nhìn từ phía `k`)

Ngược lại: lớp `d` có bao nhiêu nước `≤ k`? Đó là số `m ∈ {d, d+26, …} ≤ k`, tức `available[d] = (k − d) // 26 + 1` nếu `d ≤ k`, ngược lại `0`. Điều kiện: `cnt[d] ≤ available[d]` với mọi `d`. Cách nhìn "cung ≥ cầu" này dễ tổng quát hoá (bài tập 5, biến thể).

In [13]:
class SolutionSupply:
    def canConvertString(self, s: str, t: str, k: int) -> bool:
        if len(s) != len(t):
            return False
        demand = Counter((ord(b) - ord(a)) % 26 for a, b in zip(s, t))
        for d in range(1, 26):
            supply = (k - d) // 26 + 1 if d <= k else 0
            if demand[d] > supply:
                return False
        return True

print([SolutionSupply().canConvertString(s, t, k) for s, t, k, _ in examples])
print("k=27 -> số nước mỗi lớp d=1..3:", [(27 - d) // 26 + 1 for d in (1, 2, 3)])

[True, False, True]
k=27 -> số nước mỗi lớp d=1..3: [2, 1, 1]


**Độ phức tạp:** Time `O(n + 26)`, Space `O(26)`.

## 8. TRACE từng bước

Trace cách 3 với `s = "aabcz"`, `t = "bbdda"`, `k = 30`:

| j | s[j] | t[j] | d | used[d] trước | nước dùng | ≤ k? | used[d] sau |
|---|------|------|---|---------------|-----------|------|-------------|
| 0 | a | b | 1 | 0 | 1 | ✔ | 1 |
| 1 | a | b | 1 | 1 | 27 | ✔ | 2 |
| 2 | b | d | 2 | 0 | 2 | ✔ | 1 |
| 3 | c | d | 1 | 2 | 53 | ✘ → False | – |

Hàm `trace_early` in đúng các cột này.

In [14]:
def trace_early(s, t, k):
    if len(s) != len(t):
        print("độ dài khác nhau -> False")
        return False
    used = [0] * 26
    print(f"{'j':>2} | s | t | {'d':>2} | used trước | nước | <= k?")
    for j, (a, b) in enumerate(zip(s, t)):
        d = (ord(b) - ord(a)) % 26
        if d == 0:
            print(f"{j:>2} | {a} | {b} | {d:>2} | {'-':>10} | {'-':>4} | không cần dịch")
            continue
        move = d + 26 * used[d]
        ok = move <= k
        print(f"{j:>2} | {a} | {b} | {d:>2} | {used[d]:>10} | {move:>4} | {'có' if ok else 'KHÔNG -> False'}")
        if not ok:
            return False
        used[d] += 1
    print("mọi vị trí đều có nước -> True")
    return True

print("Kết quả:", trace_early("aabcz", "bbdda", 30))
print()
print("Kết quả:", trace_early("input", "ouput", 9))

 j | s | t |  d | used trước | nước | <= k?
 0 | a | b |  1 |          0 |    1 | có
 1 | a | b |  1 |          1 |   27 | có
 2 | b | d |  2 |          0 |    2 | có
 3 | c | d |  1 |          2 |   53 | KHÔNG -> False
Kết quả: False

 j | s | t |  d | used trước | nước | <= k?
 0 | i | o |  6 |          0 |    6 | có
 1 | n | u |  7 |          0 |    7 | có
 2 | p | p |  0 |          - |    - | không cần dịch
 3 | u | u |  0 |          - |    - | không cần dịch
 4 | t | t |  0 |          - |    - | không cần dịch
mọi vị trí đều có nước -> True
Kết quả: True


## 9. Code nộp LeetCode

Bản nộp chính (cách 3), đúng chữ ký `canConvertString(self, s: str, t: str, k: int) -> bool`:

In [15]:
class Solution:
    def canConvertString(self, s: str, t: str, k: int) -> bool:
        if len(s) != len(t):
            return False
        used = [0] * 26
        for a, b in zip(s, t):
            d = (ord(b) - ord(a)) % 26          # độ dịch vòng cần thiết
            if d:
                if d + 26 * used[d] > k:        # nước nhỏ nhất còn trống của lớp d
                    return False
                used[d] += 1
        return True

for s, t, k, want in examples:
    got = Solution().canConvertString(s, t, k)
    print(s, t, k, "->", got, "OK" if got == want else "SAI")

input ouput 9 -> True OK
abc bcd 10 -> False OK
aab bbb 27 -> True OK


## 10. Vì sao thuật toán đúng

**Điều kiện cần.** Vị trí cần độ dịch `d > 0` chỉ có thể được xử lý bởi nước `i` với `i ≡ d (mod 26)` (vì dịch `i` lần = dịch `i mod 26` lần; và mỗi vị trí chỉ được chọn một lần nên phải được dịch đúng trong **một** nước). Các vị trí khác nhau dùng nước khác nhau. Nếu có `c` vị trí cần độ dịch `d`, chúng chiếm `c` phần tử **phân biệt** của dãy `d, d+26, d+52, …`; phần tử lớn nhất trong `c` phần tử phân biệt của dãy tăng này ít nhất là phần tử thứ `c`, tức `d + 26(c − 1)`. Vậy cần `d + 26(c−1) ≤ k`.

**Điều kiện đủ.** Nếu mọi lớp thoả, gán cho các vị trí lớp `d` đúng các nước `d, d+26, …, d+26(c−1)` (đều `≤ k`). Các lớp khác nhau có số dư khác nhau nên không trùng nước; vị trí `d = 0` không dùng nước nào; các nước còn lại "không làm gì". ⇒ Biến đổi thành công.

**Độ dài khác nhau:** thao tác không thay đổi độ dài ⇒ không thể. Vì vậy thuật toán đúng.

## 11. Phân tích Time / Space

Gọi `n = len(s)` (khi bằng `len(t)`).

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. Cấp phát bằng tập "đã dùng" | `O(n²/26)` worst | `O(n)` | worst: mọi vị trí cùng độ dịch |
| 2. Đếm rồi kiểm tra | `O(n + 26)` | `O(26)` | |
| 3. Kiểm tra ngay (dừng sớm) | `O(n)` | `O(26)` | **bản nộp** |
| 4. Cung ≥ cầu | `O(n + 26)` | `O(26)` | dễ tổng quát hoá |

`k` tới `10⁹` không ảnh hưởng độ phức tạp vì ta không bao giờ duyệt qua từng nước đi.

## 12. Bộ test

Ví dụ đề, edge case (độ dài khác nhau, `k = 0`, `s == t`, vòng `'z' → 'a'`, độ dịch 25, đúng biên `d + 26(c−1) == k`), random seed cố định so với baseline, và test lớn `n = 10⁵`.

In [16]:
METHODS = {
    "brute": brute_assign,
    "count_then_check": count_then_check,
    "SolutionCount": lambda s, t, k: SolutionCount().canConvertString(s, t, k),
    "SolutionEarly": lambda s, t, k: SolutionEarly().canConvertString(s, t, k),
    "SolutionSupply": lambda s, t, k: SolutionSupply().canConvertString(s, t, k),
    "Solution": lambda s, t, k: Solution().canConvertString(s, t, k),
}
edge = [("ab", "abc", 100, False), ("abc", "ab", 100, False), ("a", "a", 0, True), ("a", "b", 0, False),
        ("a", "b", 1, True), ("z", "a", 1, True), ("z", "a", 0, False), ("a", "z", 24, False),
        ("a", "z", 25, True), ("aa", "bb", 26, False), ("aa", "bb", 27, True), ("aaa", "bbb", 53, True),
        ("aaa", "bbb", 52, False), ("abc", "abc", 0, True), ("atmtxzjkz", "tvbtjhvjd", 35, False)]
for s, t, k, want in examples + edge:
    for name, f in METHODS.items():
        got = f(s, t, k)
        assert got == want, (name, s, t, k, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [17]:
rng = random.Random(1540)
for _ in range(2000):
    n = rng.randint(1, 30)
    alpha = rng.choice(["ab", "abc", string.ascii_lowercase])
    s = "".join(rng.choice(alpha) for _ in range(n))
    if rng.random() < 0.1:
        t = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 30)))
    else:
        t = "".join(rng.choice(alpha) for _ in range(n))
    k = rng.choice([rng.randint(0, 30), rng.randint(0, 200), rng.randint(0, 10**9)])
    ref = brute_assign(s, t, k)
    for name, f in METHODS.items():
        assert f(s, t, k) == ref, (name, s, t, k)
print("2000 bộ ngẫu nhiên (seed 1540): mọi cách khớp baseline")

2000 bộ ngẫu nhiên (seed 1540): mọi cách khớp baseline


In [18]:
rng2 = random.Random(7)
big_s = "".join(rng2.choice(string.ascii_lowercase) for _ in range(10**5))
big_t = "".join(rng2.choice(string.ascii_lowercase) for _ in range(10**5))
same_d_s, same_d_t = "a" * 10**5, "b" * 10**5
for name in ["count_then_check", "SolutionEarly", "SolutionSupply", "Solution"]:
    t0 = time.perf_counter()
    f = METHODS[name]
    res = [f(big_s, big_t, 10**9), f(big_s, big_t, 10**5), f(same_d_s, same_d_t, 1 + 26 * (10**5 - 1)),
           f(same_d_s, same_d_t, 26 * (10**5 - 1))]
    print(f"{name:16s}: {res} ({time.perf_counter() - t0:.3f}s)")
    assert res[2] is True and res[3] is False

count_then_check: [True, False, True, False] (0.032s)


SolutionEarly   : [True, False, True, False] (0.054s)
SolutionSupply  : [True, False, True, False] (0.038s)


Solution        : [True, False, True, False] (0.053s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra độ dài

`zip` dừng ở chuỗi ngắn hơn ⇒ phần thừa bị bỏ qua, trả `True` sai.

In [19]:
def wrong_no_len(s, t, k):
    used = [0] * 26                                   # SAI: thiếu if len(s) != len(t)
    for a, b in zip(s, t):
        d = (ord(b) - ord(a)) % 26
        if d:
            if d + 26 * used[d] > k:
                return False
            used[d] += 1
    return True

for s, t, k in [("ab", "abc", 100), ("abcd", "ab", 5)]:
    print(s, t, k, "| sai:", wrong_no_len(s, t, k), "| đúng:", Solution().canConvertString(s, t, k))

ab abc 100 | sai: True | đúng: False
abcd ab 5 | sai: True | đúng: False


Sửa: `if len(s) != len(t): return False` ngay đầu hàm.

### Lỗi 2 — Không lấy modulo 26 khi tính độ dịch

`ord(b) − ord(a)` âm khi phải "vòng" (`'z' → 'a'` cần dịch 1, nhưng hiệu là −25). Không modulo thì chỉ số âm truy cập nhầm ô, hoặc điều kiện so sánh sai.

In [20]:
def wrong_no_mod(s, t, k):
    if len(s) != len(t):
        return False
    cnt = Counter()
    for a, b in zip(s, t):
        d = ord(b) - ord(a)                         # SAI: thiếu % 26
        if d:
            cnt[d] += 1
            if d + 26 * (cnt[d] - 1) > k:
                return False
    return True

for s, t, k in [("z", "a", 0), ("zz", "aa", 1), ("c", "a", 3)]:
    print(s, t, k, "| sai:", wrong_no_mod(s, t, k), "| đúng:", Solution().canConvertString(s, t, k))

z a 0 | sai: True | đúng: False
zz aa 1 | sai: True | đúng: False
c a 3 | sai: True | đúng: False


Sửa: `d = (ord(b) − ord(a)) % 26`.

### Lỗi 3 — Coi vị trí không cần dịch (`d = 0`) cũng tốn một nước

Vị trí `d = 0` cứ để yên. Nếu đếm nó vào lớp 0, nhiều ký tự giống nhau sẽ đòi nước 26, 52, … vô lý.

In [21]:
def wrong_count_zero(s, t, k):
    if len(s) != len(t):
        return False
    used = [0] * 26
    for a, b in zip(s, t):
        d = (ord(b) - ord(a)) % 26
        move = (d if d else 26) + 26 * used[d]     # SAI: d = 0 cũng xin nước
        if move > k:
            return False
        used[d] += 1
    return True

for s, t, k in [("aa", "aa", 0), ("input", "ouput", 9)]:
    print(s, t, k, "| sai:", wrong_count_zero(s, t, k), "| đúng:", Solution().canConvertString(s, t, k))

aa aa 0 | sai: False | đúng: True
input ouput 9 | sai: False | đúng: True


Sửa: `if d == 0: continue`.

### Lỗi 4 — Chỉ kiểm tra `max(d) ≤ k` (quên mỗi nước chỉ dùng một lần)

Nhiều vị trí cùng độ dịch không thể dùng chung một nước.

In [22]:
def wrong_max_only(s, t, k):
    if len(s) != len(t):
        return False
    return all((ord(b) - ord(a)) % 26 <= k for a, b in zip(s, t))    # SAI

for s, t, k, want in examples:
    print(s, t, k, "| sai:", wrong_max_only(s, t, k), "| đúng:", want)

input ouput 9 | sai: True | đúng: True
abc bcd 10 | sai: True | đúng: False
aab bbb 27 | sai: True | đúng: True


Sửa: đếm số lần mỗi độ dịch xuất hiện và kiểm tra `d + 26·(c − 1) ≤ k`.

### Lỗi 5 — Lệch một: dùng `d + 26·c` thay vì `d + 26·(c − 1)`

Lần đầu tiên dùng nước `d` (không cộng 26). Viết `d + 26 * cnt[d]` **sau khi** đã tăng `cnt[d]` là lệch một vòng.

In [23]:
def wrong_off_by_one(s, t, k):
    if len(s) != len(t):
        return False
    cnt = [0] * 26
    for a, b in zip(s, t):
        d = (ord(b) - ord(a)) % 26
        if d:
            cnt[d] += 1
            if d + 26 * cnt[d] > k:                  # SAI: phải là cnt[d] - 1
                return False
    return True

for s, t, k, want in examples:
    print(s, t, k, "| sai:", wrong_off_by_one(s, t, k), "| đúng:", want)

input ouput 9 | sai: False | đúng: True
abc bcd 10 | sai: False | đúng: False
aab bbb 27 | sai: False | đúng: True


Sửa: kiểm tra `d + 26 * used[d]` **trước** khi tăng (như bản nộp), hoặc `d + 26 * (cnt[d] − 1)` sau khi tăng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Thao tác thứ `i` có độ lớn `i`" + phép toán vòng (mod m)** ⇒ phân lớp các thao tác theo `i mod m`; mỗi lớp là cấp số cộng.
- "Mỗi phần tử dùng một tài nguyên **riêng**" ⇒ đếm cầu theo lớp, so với cung (`(k − d) // m + 1`).
- Luôn lấy `% 26` cho hiệu chữ cái; Python xử lý số âm đúng.
- Kiểm tra độ dài trước khi `zip`.
- Bài cùng họ: *Shifting Letters* (848), *Determine if String Halves Are Alike*, *Minimum Number of Steps to Make Two Strings Anagram* (1347), *Task Scheduler* (621 — tư duy "khe trống").

## 15. Biến thể của bài

1. **Tìm `k` nhỏ nhất** để biến đổi được: `max_d (d + 26·(cnt[d] − 1))` (bài tập 4).
2. **Được dịch ngược** (lùi chữ cái) hoặc **cả hai chiều**: vị trí có thể dùng lớp `d` hoặc lớp `26 − d` ⇒ thành bài phân phối/ghép cặp giữa hai lớp.
3. **Mỗi nước được chọn tối đa `p` chỉ số**: cung mỗi lớp nhân `p`: `p · ((k − d)//26 + 1)`.
4. **Bảng chữ cái khác kích thước `m`**: thay 26 bằng `m`.
5. **Trả về phương án cụ thể** (nước nào cho vị trí nào): bài tập 5.

In [24]:
def variant_p_per_move(s, t, k, p):
    """Biến thể 3: mỗi nước đi được dịch tối đa p chỉ số (mỗi chỉ số vẫn chỉ 1 lần)."""
    if len(s) != len(t):
        return False
    demand = Counter((ord(b) - ord(a)) % 26 for a, b in zip(s, t))
    return all(demand[d] <= p * ((k - d) // 26 + 1 if d <= k else 0) for d in range(1, 26))

print(variant_p_per_move("abc", "bcd", 10, 1), variant_p_per_move("abc", "bcd", 10, 3),
      variant_p_per_move("aab", "bbb", 1, 2))

False True True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Cấp phát bằng tập | tìm nước nhỏ nhất ≡ d chưa dùng | `O(n²/26)` | `O(n)` | kiểm chứng, dễ nghĩ |
| 2. Đếm rồi kiểm tra | nước lớn nhất lớp d = `d + 26(c−1)` | `O(n)` | `O(26)` | gọn, rõ |
| 3. Kiểm tra ngay | như 2, dừng sớm | `O(n)` | `O(26)` | **bản nộp** |
| 4. Cung ≥ cầu | số nước khả dụng mỗi lớp | `O(n)` | `O(26)` | tổng quát hoá biến thể |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `canConvertString("abc", "bcd", 27)` và `canConvertString("abc", "bcd", 53)` trả về gì?

**Bài 2 (tính tay):** Với `s = "aaz"`, `t = "bbb"`, `k` nhỏ nhất để trả `True` là bao nhiêu?

**Bài 3 (điền TODO):** Hoàn thiện `ex3_can(s, t, k)` theo cách 2 (đếm rồi kiểm tra).

**Bài 4 (tự viết lại):** Viết `ex4_min_k(s, t)` trả về `k` nhỏ nhất (hoặc `-1` nếu độ dài khác nhau).

**Bài 5 (mini challenge):** Viết `ex5_plan(s, t, k)` trả về danh sách `moves` với `moves[j]` = số thứ tự nước dùng cho vị trí `j` (`0` nếu không cần dịch); các giá trị khác 0 phải **phân biệt**, `≤ k`, và áp dụng xong ra `t`. Trả `None` nếu không thể.

In [25]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_can(s: str, t: str, k: int) -> bool:
    if len(s) != len(t):
        return False
    cnt = [0] * 26
    # TODO: đếm độ dịch của từng vị trí vào cnt
    # TODO: với d = 1..25, nếu cnt[d] > 0 và d + 26 * (cnt[d] - 1) > k thì False
    return True

In [26]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_min_k(s: str, t: str) -> int:
    # TODO
    raise NotImplementedError

def ex5_plan(s: str, t: str, k: int):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba vị trí cùng `d = 1`. (2) Cần các nước 1, 27, 53. (3) `k = 27` → `False`; `k = 53` → `True`.

**Bài 2.** (1) Độ dịch: `a→b` = 1, `a→b` = 1, `z→b` = 2. (2) Lớp 1 cần nước 1 và 27; lớp 2 cần nước 2. (3) `k` nhỏ nhất = 27.

**Bài 3.** (1) `cnt[(ord(b) − ord(a)) % 26] += 1`. (2) Bỏ qua `d = 0`. (3) Trả `False` khi vượt `k`.

**Bài 4.** (1) Đếm `cnt[d]`. (2) Với mỗi `d` có `cnt[d] > 0`: ứng viên `d + 26(cnt[d] − 1)`. (3) Lấy max (mặc định 0 nếu `s == t`).

**Bài 5.** (1) Giống bản nộp, nhưng ghi lại `move` cho vị trí `j`. (2) Kiểm tra bằng cách dịch `s[j]` đúng `moves[j]` lần. (3) Kiểm tra các nước khác 0 phân biệt bằng `len(set(...))`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
a1 = (Solution().canConvertString("abc", "bcd", 27), Solution().canConvertString("abc", "bcd", 53))
print("Bài 1:", a1)
assert a1 == (False, True)

# Bài 2
def ex4_min_k_sol(s, t):
    if len(s) != len(t):
        return -1
    cnt = [0] * 26
    for a, b in zip(s, t):
        cnt[(ord(b) - ord(a)) % 26] += 1
    return max([d + 26 * (cnt[d] - 1) for d in range(1, 26) if cnt[d]], default=0)

a2 = ex4_min_k_sol("aaz", "bbb")
print("Bài 2:", a2)
assert a2 == 27 and Solution().canConvertString("aaz", "bbb", 27) and not Solution().canConvertString("aaz", "bbb", 26)

Bài 1: (False, True)
Bài 2: 27


In [28]:
# Bài 3
def ex3_can_sol(s, t, k):
    if len(s) != len(t):
        return False
    cnt = [0] * 26
    for a, b in zip(s, t):
        cnt[(ord(b) - ord(a)) % 26] += 1
    for d in range(1, 26):
        if cnt[d] and d + 26 * (cnt[d] - 1) > k:
            return False
    return True

rng3 = random.Random(3)
for _ in range(1000):
    n = rng3.randint(1, 20)
    s = "".join(rng3.choice("abcz") for _ in range(n))
    t = "".join(rng3.choice("abcz") for _ in range(n))
    k = rng3.randint(0, 120)
    assert ex3_can_sol(s, t, k) == brute_assign(s, t, k)
print("Bài 3: OK")

Bài 3: OK


In [29]:
# Bài 4
rng4 = random.Random(4)
for _ in range(500):
    n = rng4.randint(1, 15)
    s = "".join(rng4.choice("abcxyz") for _ in range(n))
    t = "".join(rng4.choice("abcxyz") for _ in range(n))
    m = ex4_min_k_sol(s, t)
    assert Solution().canConvertString(s, t, m)
    assert m == 0 or not Solution().canConvertString(s, t, m - 1)
assert ex4_min_k_sol("ab", "abc") == -1 and ex4_min_k_sol("abc", "abc") == 0
print("Bài 4: OK, ví dụ:", [ex4_min_k_sol(s, t) for s, t, _, _ in examples])

Bài 4: OK, ví dụ: [7, 53, 27]


In [30]:
# Bài 5
def ex5_plan_sol(s, t, k):
    if len(s) != len(t):
        return None
    used = [0] * 26
    moves = []
    for a, b in zip(s, t):
        d = (ord(b) - ord(a)) % 26
        if d == 0:
            moves.append(0)
            continue
        m = d + 26 * used[d]
        if m > k:
            return None
        used[d] += 1
        moves.append(m)
    return moves

def check_plan(s, t, k, moves):
    nz = [m for m in moves if m]
    return (len(nz) == len(set(nz)) and all(1 <= m <= k for m in nz)
            and "".join(shift(c, m) for c, m in zip(s, moves)) == t)

rng5 = random.Random(5)
for _ in range(1000):
    n = rng5.randint(1, 20)
    s = "".join(rng5.choice("abyz") for _ in range(n))
    t = "".join(rng5.choice("abyz") for _ in range(n))
    k = rng5.randint(0, 100)
    plan = ex5_plan_sol(s, t, k)
    if Solution().canConvertString(s, t, k):
        assert plan is not None and check_plan(s, t, k, plan)
    else:
        assert plan is None
print("Bài 5: OK, ví dụ:", ex5_plan_sol("input", "ouput", 9), ex5_plan_sol("aab", "bbb", 27))

Bài 5: OK, ví dụ: [6, 7, 0, 0, 0] [1, 27, 0]


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def canConvertString(self, s, t, k):
        if len(s) != len(t): return False
        used = [0] * 26
        for a, b in zip(s, t):
            d = (ord(b) - ord(a)) % 26
            if d:
                if d + 26 * used[d] > k: return False   # nước nhỏ nhất còn trống của lớp d
                used[d] += 1
        return True
```

**Template "thao tác thứ i có độ lớn i, phép toán mod m":** phân lớp theo `i mod m`; lớp `d` là cấp số cộng `d, d+m, …`; lần thứ `c` dùng lớp `d` tốn `d + m(c−1)`; cung của lớp = `(k − d) // m + 1`.

**Checklist trước khi submit:**
- [ ] Kiểm tra `len(s) != len(t)`.
- [ ] Độ dịch dùng `% 26` (xử lý vòng `'z' → 'a'`).
- [ ] Bỏ qua `d = 0`.
- [ ] Công thức `d + 26 * (số lần đã dùng trước đó)` — không lệch một.
- [ ] Thử nhanh: `("abc","bcd",10) → False`, `("aab","bbb",27) → True`, `("z","a",1) → True`.